# 002 · Descriptive Statistics with statistics

> **Standard Library › Numbers & Math** · Python 3.14

The `statistics` module computes averages, spread, percentiles, correlation and simple linear regression on ordinary Python lists — no third-party package needed. It's aimed at "calculator-level" statistics; for large arrays or heavy number crunching the ecosystem uses NumPy/pandas, but for everyday reports and checks the standard library is enough.

**In this notebook you will learn:**

- averages: `mean`, `fmean`, `median` (and its variants), `mode`, `multimode`
- spread: `stdev` vs `pstdev`, `variance` vs `pvariance` — sample vs population
- percentiles with `quantiles`
- relationships: `correlation`, `covariance`, `linear_regression`
- the normal distribution with `NormalDist`

## .NET mental model

The .NET BCL has **no statistics library**. LINQ gives you `Average()`, `Sum()`, `Min()` and `Max()`, and that's it — for a median, standard deviation or percentile you either write it yourself or add a NuGet package such as **MathNet.Numerics**.

| Python `statistics` | .NET |
|---|---|
| `mean(xs)` / `fmean(xs)` | LINQ `xs.Average()` |
| `median(xs)` | — (sort and pick by hand) / MathNet `xs.Median()` |
| `stdev(xs)` / `pstdev(xs)` | — / MathNet `xs.StandardDeviation()` / `xs.PopulationStandardDeviation()` |
| `quantiles(xs, n=4)` | — / MathNet `Statistics` quantile functions |
| `correlation(x, y)` | — / MathNet `Correlation.Pearson(x, y)` |
| `linear_regression(x, y)` | — / MathNet `Fit.Line(x, y)` |
| `NormalDist(mu, sigma)` | — / MathNet `Distributions.Normal` |

```csharp
// C# — the BCL stops at the mean
double avg = values.Average();
```

## Averages: `mean`, `fmean`, `median`

In [ ]:
import statistics as st

response_ms = [120, 135, 128, 142, 2100, 131, 125]   # one slow outlier

print(st.mean(response_ms))      # pulled up by the outlier
print(st.median(response_ms))    # robust "typical" value
print(st.fmean(response_ms))     # float mean, faster

`mean` vs `fmean`:

- `mean()` computes an **exact** result and keeps the input's number type where it can (`int`, `Fraction`, `Decimal`) — so it can return an `int`.
- `fmean()` always converts to `float`; it's much faster for large inputs and supports `weights=` (3.11+).

In [ ]:
print(st.mean([2, 4, 6]), st.mean([1, 2]))             # 4 (an int!) and 1.5
print(st.fmean([2, 4, 6]))                             # 4.0

grades  = [90, 70, 80]
credits = [4, 1, 2]
print(st.fmean(grades, weights=credits))               # weighted average

With an even number of items, `median` averages the two middle values; `median_low` / `median_high` return an **actual element** instead (useful when the data isn't numeric-averagable, like dates or ranks):

In [ ]:
evens = [10, 20, 30, 40]
print(st.median(evens), st.median_low(evens), st.median_high(evens))   # 25.0 20 30

## `mode` and `multimode`

`mode` returns the most common value — and works on **any hashable** data, not just numbers. If several values tie, it returns the first one encountered; `multimode` returns all of them.

In [ ]:
sizes = ["M", "L", "M", "S", "L"]
print(st.mode(sizes))         # 'M' — first of the tied values
print(st.multimode(sizes))    # ['M', 'L']

**Tip:** for full frequency tables, `collections.Counter(...).most_common()` is the better tool *(see Standard Library › Data Structures › 002 · Counting with Counter)*.

## Spread: sample vs population

This is the part people most often get wrong.

| Function | Divides by | Use when your data is... |
|---|---|---|
| `stdev`, `variance` | *n − 1* | a **sample** drawn from a larger population (the common case) |
| `pstdev`, `pvariance` | *n* | the **entire** population |

Excel users know these as `STDEV.S` and `STDEV.P`; MathNet calls them `StandardDeviation` and `PopulationStandardDeviation`.

In [ ]:
data = [2, 4, 4, 4, 5, 5, 7, 9]

print(st.pstdev(data), st.pvariance(data))   # population: 2.0 4
print(st.stdev(data), st.variance(data))     # sample: slightly larger

⚠️ **Gotcha:** functions raise `StatisticsError` (a subclass of `ValueError`) instead of returning `NaN` for too-little data — `mean([])` fails, and so does `stdev` with fewer than two points.

In [ ]:
for func, arg in [(st.mean, []), (st.stdev, [42])]:
    try:
        func(arg)
    except st.StatisticsError as e:
        print(f"{type(e).__name__}: {e}")

## Percentiles with `quantiles`

`quantiles(data, n=4)` returns the **n − 1 cut points** that divide the data into `n` equal groups: quartiles for `n=4`, deciles for `n=10`, percentiles for `n=100`.

In [ ]:
latencies = [12, 15, 11, 30, 14, 13, 18, 22, 16, 45, 17, 19]

q1, q2, q3 = st.quantiles(latencies, n=4)
print(f"Q1={q1}  median={q2}  Q3={q3}  IQR={q3 - q1}")

p = st.quantiles(latencies, n=100)
print("p90:", round(p[89], 2), " p95:", round(p[94], 2))   # index k-1 is the k-th percentile
print("max:", max(latencies))                               # p95 > max?!

⚠️ **Gotcha:** there are several ways to compute percentiles. The default `method="exclusive"` assumes your data is a *sample* of a bigger population, so it may **extrapolate beyond the observed min/max** — that's why p95 above is larger than the largest latency. `method="inclusive"` treats the data as the whole population, always stays within the data range, and matches Excel's `PERCENTILE.INC` / NumPy's default. Results differ noticeably on small datasets:

In [ ]:
small = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(st.quantiles(small, n=4))                      # exclusive (default)
print(st.quantiles(small, n=4, method="inclusive"))
print(round(st.quantiles(latencies, n=100, method="inclusive")[94], 2))   # p95 now within the data

## Correlation and linear regression (3.10+)

```csharp
// C# with MathNet.Numerics
double r = Correlation.Pearson(x, y);
var (intercept, slope) = Fit.Line(x, y);
```

`correlation` returns Pearson's *r* (−1 … 1). `linear_regression` fits `y = slope * x + intercept` by ordinary least squares and returns a named tuple.

In [ ]:
ad_spend = [1, 2, 3, 4, 5]          # thousands
sales    = [12, 15, 21, 22, 30]     # units

print(round(st.correlation(ad_spend, sales), 4))
print(round(st.covariance(ad_spend, sales), 4))

fit = st.linear_regression(ad_spend, sales)
print(fit)
print("predicted sales at 6k:", round(fit.slope * 6 + fit.intercept, 1))

Two extras: `correlation(x, y, method="ranked")` (3.12+) computes Spearman's rank correlation, which is robust to outliers and non-linear-but-monotonic relationships; `linear_regression(x, y, proportional=True)` (3.11+) forces the line through the origin.

In [ ]:
x = [1, 2, 3, 4, 5]
y = [1, 4, 9, 16, 1000]             # monotonic, but very non-linear
print(round(st.correlation(x, y), 3), st.correlation(x, y, method="ranked"))

## `NormalDist` — the normal distribution as an object

`NormalDist(mu, sigma)` models a normal (Gaussian) distribution. You can build one from parameters or fit one to data, then ask probability questions.

| Method | Question |
|---|---|
| `cdf(x)` | P(X ≤ x) |
| `inv_cdf(p)` | the value below which a fraction *p* falls |
| `pdf(x)` | probability density at *x* |
| `zscore(x)` | how many standard deviations from the mean |
| `from_samples(data)` | fit `mu` and `sigma` to data |

In [ ]:
from statistics import NormalDist

scores = NormalDist(mu=100, sigma=15)

print(f"P(score <= 130) = {scores.cdf(130):.4f}")
print(f"P(85 < score < 115) = {scores.cdf(115) - scores.cdf(85):.4f}")   # ~68%
print(f"95th percentile = {scores.inv_cdf(0.95):.1f}")
print(f"z-score of 130 = {scores.zscore(130)}")

In [ ]:
fitted = NormalDist.from_samples(data)
print(fitted)                            # mean and *sample* stdev of the data
print(round(fitted.mean, 2), round(fitted.stdev, 2))

`NormalDist` objects also support arithmetic: adding two independent distributions or scaling by a constant gives you the resulting distribution.

In [ ]:
task_a = NormalDist(30, 4)    # minutes
task_b = NormalDist(45, 3)
total = task_a + task_b
print(total, f"P(total > 85 min) = {1 - total.cdf(85):.3f}")

**Tip:** `statistics` also works with `Fraction` and `Decimal` inputs and keeps their exactness in `mean`, `median` etc. *(see Standard Library › Numbers & Math › 003 · Decimal Arithmetic)*.

In [ ]:
from decimal import Decimal
print(repr(st.mean([Decimal("1.10"), Decimal("2.20"), Decimal("3.30")])))   # Decimal('2.2') — exact, still a Decimal

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Mean | `xs.Average()` | `statistics.mean(xs)` / `fmean(xs)` |
| Weighted mean | hand-written | `fmean(xs, weights=ws)` (3.11+) |
| Median | sort + pick / MathNet | `median(xs)` |
| Most common | `GroupBy` + `OrderByDescending` | `mode(xs)` / `multimode(xs)` |
| Sample std dev | MathNet `StandardDeviation()` | `stdev(xs)` |
| Population std dev | MathNet `PopulationStandardDeviation()` | `pstdev(xs)` |
| Quartiles / percentiles | MathNet | `quantiles(xs, n=4)` / `n=100` |
| Pearson correlation | MathNet `Correlation.Pearson` | `correlation(x, y)` (3.10+) |
| Least-squares line | MathNet `Fit.Line` | `linear_regression(x, y)` (3.10+) |
| Normal distribution | MathNet `Distributions.Normal` | `NormalDist(mu, sigma)` |

## Exercises

1. Given `orders = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 14]`, print the mean, median, all modes and the sample standard deviation (rounded to 2 places).
2. Using `quantiles`, find the interquartile range (Q3 − Q1) of `orders` and list any values more than 1.5 × IQR above Q3.
3. Fit a line to `hours = [1, 2, 3, 4, 5, 6]` vs `score = [52, 55, 61, 64, 70, 73]` and predict the score for 8 hours.
4. Exam scores follow `NormalDist(70, 10)`. What fraction of students score above 85? What score is needed to be in the top 10%?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import statistics as st
from statistics import NormalDist

# 1
orders = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 14]
print(st.mean(orders), st.median(orders), st.multimode(orders), round(st.stdev(orders), 2))
# 4.833333333333333 4.5 [5] 3.66

# 2
q1, _, q3 = st.quantiles(orders, n=4)
iqr = q3 - q1
print(iqr, [x for x in orders if x > q3 + 1.5 * iqr])   # 3.5 [14]

# 3
hours = [1, 2, 3, 4, 5, 6]
score = [52, 55, 61, 64, 70, 73]
fit = st.linear_regression(hours, score)
print(round(fit.slope * 8 + fit.intercept, 1))   # 82.2

# 4
exam = NormalDist(70, 10)
print(f"{1 - exam.cdf(85):.1%}")       # ~6.7%
print(round(exam.inv_cdf(0.90), 1))    # ~82.8
```

</details>